In [66]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict,Annotated
from langchain_core.messages import BaseMessage,HumanMessage,AIMessage
from langgraph.graph.message import add_messages
from dotenv import load_dotenv
from langchain_openai import AzureChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver

from langgraph.prebuilt import ToolNode, tools_condition
from langchain_community.tools import DuckDuckGoSearchResults
from langchain_core.tools import tool

import requests
import random

import os

In [67]:
load_dotenv()

AZURE_OPENAI_ENDPOINT = os.getenv('AZURE_OPENAI_ENDPOINT_EUS2')
AZURE_OPENAI_API_KEY = os.getenv('AZURE_OPENAI_APIKEY_EUS2')
AZURE_OPENAI_API_VERSION = os.getenv('AZURE_OPENAI_API_VERSION', '2025-04-01-preview')
LLM_DEPLOYMENT_NAME = os.getenv('LLM_DEPLOYMENT_NAME', os.getenv('MODEL_NAME'))


model = AzureChatOpenAI(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    azure_deployment=LLM_DEPLOYMENT_NAME,
    api_version=AZURE_OPENAI_API_VERSION,
    temperature=0.2,
)

In [68]:
#tools

search_tool = DuckDuckGoSearchResults(region="us", safesearch="Moderate", max_results=5)


@tool("calculator")
def calculator(first_num:float,second_num:float,operation:str)->str:
    """Perform a basic arithmetic operation on two numbers.
    supported operations: add,sub,mul,div"""
    if operation == 'add':
        return str(first_num + second_num)
    elif operation == 'subtract':
        return str(first_num - second_num)
    elif operation == 'multiply':
        return str(first_num * second_num)
    elif operation == 'divide':
        if second_num == 0:
            return "Error: Division by zero"
        return str(first_num / second_num)
    else:
        return "Error: Invalid operation. Supported operations are add, subtract, multiply, divide."
    


@tool("joke_generator")

def joke_generator(user_input:str)->str:
    """Generate a joke based on the user's input."""
    prompt = f"Explain the joke {user_input} in detail"
    response = model.invoke(prompt).content
    return {
        'explaination': response,
    }
    

In [69]:
tools = [search_tool,calculator,joke_generator]

llm_with_tools = model.bind_tools(tools)

In [70]:
#state
class ChatState(TypedDict):
    messages:Annotated[list[BaseMessage],add_messages]

In [71]:
def chat_node(state:ChatState)->ChatState:
    """LLM node thast may answer or request a tool call"""
    messages = state['messages']
    response = llm_with_tools.invoke(messages)
    return {"messages":[response]}
    
tool_node = ToolNode(tools)#execute tool calls

In [72]:
graph = StateGraph(ChatState)
graph.add_node("chat_node", chat_node)
graph.add_node("tools",tool_node)


In [73]:
graph.add_edge(START,"chat_node")
graph.add_conditional_edges("chat_node",tools_condition)
graph.add_edge("tools", "chat_node")
graph.add_edge("chat_node",END)

In [74]:
checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)
initial_state = {"messages":[HumanMessage(content=" joke on pizza")]}
config1 = {"configurable": {"thread_id": "1"}}


result = workflow.invoke(initial_state,config=config1)

print(result['messages'][-1].content)


Sure — here’s a pizza joke:

**Why did the pizza maker go broke?**  
Because he just couldn’t make enough **dough**.

If you want, I can give you:
- a **shorter** pizza joke
- a **cheesier** one
- or explain a pizza joke/meme you saw


Failed to batch ingest runs: langsmith.utils.LangSmithConnectionError: Connection error caused failure to POST https://api.smith.langchain.com/runs/batch in LangSmith API. Please confirm your internet connection. SSLError(MaxRetryError("HTTPSConnectionPool(host='api.smith.langchain.com', port=443): Max retries exceeded with url: /runs/batch (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1028)')))"))
Content-Length: 5645
API Key: lsv2_********************************************1a
post: trace=1bc08500-68db-47a0-920f-738681cf5197,id=1bc08500-68db-47a0-920f-738681cf5197; trace=1bc08500-68db-47a0-920f-738681cf5197,id=c4df28d9-a925-4a8a-9a6f-350db7dbb739; trace=1bc08500-68db-47a0-920f-738681cf5197,id=33982acc-9d4b-4333-833b-5f1d9354399e


In [65]:
workflow.get_state(config1)


StateSnapshot(values={'messages': [HumanMessage(content=' joke on pizza', additional_kwargs={}, response_metadata={}, id='15e29faf-f02c-42ff-86db-02c6d8489a23'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_Kyvb7pzqrWaVCRQzQowZ9ghl', 'function': {'arguments': '{"user_input":"pizza"}', 'name': 'joke_generator'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 20, 'prompt_tokens': 225, 'total_tokens': 245, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-5.4-2026-03-05', 'system_fingerprint': None, 'id': 'chatcmpl-DUVG00n0kPHBWGdikwg82gliklZOE', 'service_tier': 'default', 'prompt_filter_results': [{'prompt_index': 0, 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'}, 'jailbreak': {'detected': False, 'filtered': False}, 

In [75]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'messages': [HumanMessage(content=' joke on pizza', additional_kwargs={}, response_metadata={}, id='1bb36772-7d16-4d70-8212-c2fc4f2c0a35'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_3ocpHu2Wr9uigvuf7SQXpdwS', 'function': {'arguments': '{"user_input":"pizza"}', 'name': 'joke_generator'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 20, 'prompt_tokens': 225, 'total_tokens': 245, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-5.4-2026-03-05', 'system_fingerprint': None, 'id': 'chatcmpl-DUVTpHcwUyocEMwbdJxClbt2is1ri', 'service_tier': 'default', 'prompt_filter_results': [{'prompt_index': 0, 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'}, 'jailbreak': {'detected': False, 'filtered': False},

Failed to batch ingest runs: langsmith.utils.LangSmithConnectionError: Connection error caused failure to POST https://api.smith.langchain.com/runs/batch in LangSmith API. Please confirm your internet connection. SSLError(MaxRetryError("HTTPSConnectionPool(host='api.smith.langchain.com', port=443): Max retries exceeded with url: /runs/batch (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1028)')))"))
Content-Length: 46966
API Key: lsv2_********************************************1a
post: trace=1bc08500-68db-47a0-920f-738681cf5197,id=96224cba-ea5c-4976-90e0-ff5a8f121cc2; trace=1bc08500-68db-47a0-920f-738681cf5197,id=5510f42a-00a4-4506-9275-94a87ad8b3b9; trace=1bc08500-68db-47a0-920f-738681cf5197,id=aaef227e-afbc-4fd5-a4cb-45434968f939; trace=1bc08500-68db-47a0-920f-738681cf5197,id=6ffdb25d-f786-46d1-aa00-d20ade025775; trace=1bc08500-68db-47a0-920f-738681cf5197,id=7b8d3423-c43f-4e4